# Provider adapters, structured output, and guardrails: practice

Wrap compatible async clients as model adapters → configure agents → define NameCheckOutput → run the checking agent → map its decision to a tripwire.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [2_openai/3_lab3.ipynb](../../../../../2_openai/3_lab3.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configure a real SDK agent and inspect its tool schema

Agent holds instructions/model/tools. The function_tool decorator constructs a schema from the signature and docstring. Constructing this configuration does not run the agent or contact a provider.

**Predict:** Which argument names will appear in the registered tool schema?

In [ ]:
from agents import Agent, function_tool, set_tracing_disabled
set_tracing_disabled(True)
@function_tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b
agent = Agent(name="Practice teacher", instructions="Use add_numbers for addition.",
              model="gpt-4o-mini", tools=[add_numbers])
print(agent.name, agent.tools[0].name)
print(sorted(agent.tools[0].params_json_schema["properties"]))
assert sorted(agent.tools[0].params_json_schema["properties"]) == ["a", "b"]

**Check:** a and b. The schema advertises the function; Runner executes the model/tool workflow when deliberately invoked.

**Change one thing:** Change the docstring and inspect the tool description. Find Runner.run in the source: that is where the interaction loop actually begins.

## 2. Provider endpoint and model selection

A compatible client can target different services. The endpoint selects the provider and its credentials; the model identifier selects a model there. Compatibility does not guarantee every feature works.

**Predict:** Does changing only the variable name change the provider?

In [ ]:
# Non-secret connection descriptions only; no clients or keys are created.
providers = {
    "openai": {"key_variable": "OPENAI_API_KEY", "endpoint": "default"},
    "gemini": {"key_variable": "GOOGLE_API_KEY", "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai/"},
    "ollama": {"key_variable": None, "endpoint": "http://localhost:11434/v1"},
}
for provider, settings in providers.items():
    print(provider, settings["endpoint"], settings["key_variable"])
assert providers["gemini"]["key_variable"] == "GOOGLE_API_KEY"

**Check:** No. The configured endpoint/client and model select the service. Local Ollama still needs its service and downloaded model.

**Change one thing:** Add a provider from the linked source. Copy its endpoint and variable name, never a credential value.

## 3. Structured output validates shape

Pydantic models name the fields you expect and check constraints. This helps callers use results reliably, but a valid field can still contain a false claim.

**Predict:** Will confidence=2 pass the constraint?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class Report(BaseModel):
    summary: str
    confidence: float = Field(ge=0, le=1)
report = Report(summary="Fictional result", confidence=0.8)
print(report.model_dump())
try:
    Report(summary="Bad shape", confidence=2)
except ValidationError:
    print("Invalid confidence rejected")
else:
    raise AssertionError("Constraint was not enforced")

**Check:** No. The value must be between zero and one. Validation checks the constraint, not the truth of the summary.

**Change one thing:** Add a field named follow_up_questions with type list[str]. Construct a valid object using it.

## 4. A guardrail makes a stop/continue decision

An input guardrail checks a request before later work. The tripwire is a boolean. This deterministic example teaches the branch; it is not the model-based name detector in the source.

**Predict:** Does Alice trigger this demonstration guardrail?

In [ ]:
from pydantic import BaseModel
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str
def check_name(message):
    found = "Alice" in message  # Deliberately limited teaching rule.
    return NameCheckOutput(is_name_in_message=found, name="Alice" if found else "")
for request in ["Send from Alice", "Send from Head of Business Development"]:
    result = check_name(request)
    tripwire_triggered = result.is_name_in_message
    print("stop" if tripwire_triggered else "continue")
assert check_name("Send from Alice").is_name_in_message

**Check:** Yes for Alice, no for the title. A boolean decision does not establish that every personal name is detected.

**Change one thing:** Try another name and observe the limitation. Describe how you would evaluate false positives and missed names.

## 5. Agent as a tool vs. handoff

A tool returns its result to the calling agent. A handoff transfers control to the receiving agent. This trace uses ordinary Python to make that distinction visible.

**Predict:** After the tool finishes, who produces the next step?

In [ ]:
def specialist_tool(topic):
    return f"Specialist result for {topic}"
tool_trace = ["manager starts", specialist_tool("Python"), "manager resumes"]
handoff_trace = ["manager starts", "manager hands off", "specialist controls remaining reply"]
print("Tool:", tool_trace)
print("Handoff:", handoff_trace)
assert tool_trace[-1] == "manager resumes"

**Check:** The manager resumes after a tool. The specialist takes control after a handoff.

**Change one thing:** Sketch which pattern fits a writer consulting a researcher, and which fits routing a customer to support.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.